# 04 - Agent Workflow Offline Demo

This notebook runs the full Member 2 workflow with mocked dependencies. It does not use OpenRouter credits and does not perform live web search.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.agent.agent_workflow import AgentDependencies, answer_labor_law_question
from src.agent.llm_client import MockLLMClient
from src.agent.tools.legal_retrieval_tool import LegalRetrievalAdapter
from src.agent.tools.web_search_tool import DuckDuckGoSearchTool
from src.agent.tools.web_scraper_tool import WebScraperTool

## Scenario

We will use an unpaid salary case, because it is easy to understand and appears in the project proposal.

In [ ]:
question = "My company has not paid my salary for 2 months and asks me to sign a resignation letter. What should I do?"
question

## Mock the Final LLM Response

The workflow calls the LLM twice: first to identify legal issues, then to synthesize final JSON. `MockLLMClient` returns deterministic JSON so we can learn the workflow offline.

In [ ]:
mock_final_response = {
    "case_summary": "The user reports unpaid salary and pressure to sign a resignation letter.",
    "legal_issues": ["unpaid salary", "forced resignation"],
    "legal_basis": [
        {
            "document_name": "Labor Code",
            "article": "Article 94",
            "effective_status": "Effective",
            "excerpt": "Employers must pay wages fully and on time.",
            "source": "mock://labor-code/article-94",
        }
    ],
    "practical_references": [
        {
            "title": "Unpaid Salary Case",
            "url": "https://example.com/unpaid-salary",
            "summary": "A worker collected evidence and sent written requests.",
        }
    ],
    "recommendations": [
        {
            "option": "Collect evidence and send a written request",
            "when_to_use": "Use this before escalating the dispute.",
            "suggested_steps": ["Collect payslips", "Save messages", "Send a written salary request"],
        }
    ],
    "limitations": "Decision support only; contact a lawyer or authority for official advice.",
}

llm = MockLLMClient(json.dumps(mock_final_response, ensure_ascii=False))

In [ ]:
legal_tool = LegalRetrievalAdapter(
    mock_results=[
        {
            "document_name": "Labor Code",
            "article": "Article 94",
            "effective_status": "Effective",
            "excerpt": "Employers must pay wages fully and on time.",
            "source": "mock://labor-code/article-94",
        }
    ]
)

search_tool = DuckDuckGoSearchTool(
    provider=lambda query, max_results: [
        {
            "title": "Unpaid Salary Case",
            "url": "https://example.com/unpaid-salary",
            "snippet": "A worker collected evidence.",
        }
    ]
)

scraper_tool = WebScraperTool(
    fetcher=lambda url: "<title>Unpaid Salary Case</title><article>A worker collected payslips and messages.</article>"
)

dependencies = AgentDependencies(
    llm_client=llm,
    legal_tool=legal_tool,
    search_tool=search_tool,
    scraper_tool=scraper_tool,
)

In [ ]:
answer = answer_labor_law_question(question, dependencies)
print(json.dumps(answer, indent=2, ensure_ascii=False))

## Failure Behavior

The workflow should still return JSON when a tool fails or when the LLM returns an unexpected shape.

In [ ]:
bad_llm = MockLLMClient('{"unexpected": "shape"}')
failure_dependencies = AgentDependencies(
    llm_client=bad_llm,
    legal_tool=LegalRetrievalAdapter(mock_results=[]),
    search_tool=DuckDuckGoSearchTool(provider=lambda query, max_results: []),
    scraper_tool=scraper_tool,
)

fallback_answer = answer_labor_law_question(question, failure_dependencies)
print(json.dumps(fallback_answer, indent=2, ensure_ascii=False))

## What You Should Understand Now

Dependency injection lets you test the whole Agent workflow without OpenRouter, without live web search, and without Member 1's real retrieval system.